In [1]:
# importing libraries

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

In [2]:
train_df = pd.read_pickle("../data/train.pkl")
test_df = pd.read_pickle("../data/test.pkl")

In [3]:
x_train = train_df.drop(columns=["isFraud","TransactionDT"])
y_train = train_df["isFraud"]

x_test = test_df.drop(columns=["TransactionDT"])

In [4]:
INT_TYPES = [np.int8, np.int16, np.int32, np.int64]
UINT_TYPES = [np.uint8, np.uint16, np.uint32, np.uint64]
COL_AMOUNT =  'TransactionAmt'
IGNORE_COLUMNS = [
    COL_AMOUNT
]


def downcast_int(df, col, min_val, max_val):    
    if min_val < 0:
        for i, dtype in enumerate(INT_TYPES):
            if (i == 3) or (np.iinfo(dtype).min < min_val and max_val < np.iinfo(dtype).max):
                df[col] = df[col].astype(dtype); break
    else:
        for i, dtype in enumerate(UINT_TYPES):
            if (i == 3) or (max_val < np.iinfo(dtype).max):
                df[col] = df[col].astype(dtype); break


def mem_usg_mb(df):
    return df.memory_usage().sum() / 1024**2


# original: https://www.kaggle.com/mjbahmani/reducing-memory-size-for-ieee
def reduce_mem_usage(df):
    init_mem_usg = mem_usg_mb(df)
    NAlist = [] # Keeps track of columns that have missing values filled in. 
    for col in df.columns:
        if col in IGNORE_COLUMNS or df[col].dtype == object:
            continue
        dtype_before = df[col].dtype
        min_val, max_val = df[col].min(), df[col].max()
        # Integer does not support NA, therefore, NA needs to be filled
        if not np.isfinite(df[col]).all(): 
            NAlist.append(col)
            # df[col].fillna(-32222, inplace=True) # changed here mn-1 -> -32222
        # test if column can be converted to an integer
        # https://stackoverflow.com/questions/21583758/how-to-check-if-a-float-value-is-a-whole-number       
        if (df[col] % 1 == 0).all():
            downcast_int(df, col, min_val, max_val)  
        else:
            df[col] = df[col].astype(np.float32)
        print("{}, range: ({}, {}), dtype: {} -> {}".format(
            col, min_val, max_val, dtype_before, df[col].dtype))
    print("\n *** DONE ***")
    mem_usg = mem_usg_mb(df) 
    print("Memory usage: %.02f -> %.02f MB" % (init_mem_usg, mem_usg))
    print("This is {:.02f}% of the initial size".format(100 * mem_usg / init_mem_usg))
    return df, NAlist

In [5]:
x_train_mem = x_train.copy()
x_test_mem = x_test.copy()

x_train_mem , na_train = reduce_mem_usage(x_train_mem)
x_test_mem , na_test = reduce_mem_usage(x_test_mem)

TransactionID, range: (2987000, 3577539), dtype: int64 -> uint32
card1, range: (1000, 18396), dtype: int64 -> uint16
card2, range: (100.0, 600.0), dtype: float64 -> float32
card3, range: (100.0, 231.0), dtype: float64 -> float32
card5, range: (100.0, 237.0), dtype: float64 -> float32
addr1, range: (100.0, 540.0), dtype: float64 -> float32
addr2, range: (10.0, 102.0), dtype: float64 -> float32
dist1, range: (0.0, 10286.0), dtype: float64 -> float32
dist2, range: (0.0, 11623.0), dtype: float64 -> float32
C1, range: (0.0, 4685.0), dtype: float64 -> uint16
C2, range: (0.0, 5691.0), dtype: float64 -> uint16
C3, range: (0.0, 26.0), dtype: float64 -> uint8
C4, range: (0.0, 2253.0), dtype: float64 -> uint16
C5, range: (0.0, 349.0), dtype: float64 -> uint16
C6, range: (0.0, 2253.0), dtype: float64 -> uint16
C7, range: (0.0, 2255.0), dtype: float64 -> uint16
C8, range: (0.0, 3331.0), dtype: float64 -> uint16
C9, range: (0.0, 210.0), dtype: float64 -> uint8
C10, range: (0.0, 3257.0), dtype: float

In [6]:
print(x_train_mem.memory_usage(deep=True).sum() / 1024**3)
print(x_test_mem.memory_usage(deep=True).sum() / 1024**3)

1.8909522918984294
1.6025949530303478


In [7]:
x_train_mem.columns.equals(x_test_mem.columns) , x_train_mem.dtypes.value_counts()

(True,
 float32    398
 object      36
 uint8       22
 uint16      15
 uint32       2
 int8         2
 float64      1
 Name: count, dtype: int64)

In [8]:
numeric_cols = x_train_mem.select_dtypes(include=["int8" , "float32" , "float64" , "uint8" , "uint16" , "uint32"]).columns.tolist()
categorical_cols = x_train_mem.select_dtypes(include=["object"]).columns.tolist()

In [9]:
# Initializing PIPELINE

In [10]:
# NUMERICAL

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median' , add_indicator=True))

])

In [11]:
# CATEGORICAL

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant' , fill_value="Missing")) ,
    ('encoder' , OneHotEncoder(handle_unknown='ignore' , sparse_output=True , dtype=np.float32))
])

In [12]:
preprocessor = ColumnTransformer([
        ('num' , numeric_pipeline , numeric_cols) ,
        ('cat' , categorical_pipeline , categorical_cols)
] , sparse_threshold=1.0)

In [13]:
x_train_transformed = preprocessor.fit_transform(x_train_mem , y_train)

In [14]:
x_test_transformed = preprocessor.transform(x_test_mem)

In [15]:
import joblib
import os

os.makedirs("../models/",exist_ok=True)

In [16]:
lgb_test_predictions = []

lgb_seeds = [123 , 2024, 777]

for seed in lgb_seeds:
    print(f"\nLightGBM SEED-{seed}")

    lgb_model = LGBMClassifier( objective="binary" , random_state=seed , n_jobs=-1 , max_depth=-1 , n_estimators=100 , learning_rate = 0.05 ,subsample=0.8 ,min_child_weight = 5, 
                num_leaves=31 , colsample_bytree = 0.8, reg_alpha = 0.0, reg_lambda = 0.0 , min_child_samples=20)
    
    lgb_model.fit(x_train_transformed , y_train)

    lgb_model.booster_.save_model(f"../models/lgb_seed_{seed}.txt")
    print(f"Saved LightGBM seed-{seed}")

    test_pred = lgb_model.predict_proba(x_test_transformed)[:,1]
    lgb_test_predictions.append(test_pred)

lgb_test_pred = np.mean(lgb_test_predictions, axis=0)    


LightGBM SEED-123
[LightGBM] [Info] Number of positive: 20663, number of negative: 569877
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 65.883139 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 85715
[LightGBM] [Info] Number of data points in the train set: 590540, number of used features: 23223
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034990 -> initscore=-3.317076
[LightGBM] [Info] Start training from score -3.317076
Saved LightGBM seed-123

LightGBM SEED-2024
[LightGBM] [Info] Number of positive: 20663, number of negative: 569877
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 67.646493 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 85856
[LightGBM] [Info] Number of data point

In [17]:
lgb_test_output = pd.DataFrame({
    "TransactionID" : test_df["TransactionID"] , 
    "lgb_seed_123" : lgb_test_predictions[0] ,
    "lgb_seed_2024" : lgb_test_predictions[1] ,
    "lgb_seed_777" : lgb_test_predictions[2] ,
    "lightgbm_test" : lgb_test_pred
})

lgb_test_output.to_csv("../outputs/lightgbm_test_predictions.csv" , index=False)

In [18]:
joblib.dump(preprocessor , "../models/lgb_preprocessor.pkl")

['../models/lgb_preprocessor.pkl']